In [1]:
from cryptography.fernet import Fernet

In [2]:
#generate and save fernet key
key = Fernet.generate_key()
key

b'NwMAISFRTA7chl4w7f8YCbZjL4bR9NuHBPkr_0UcX1s='

In [3]:
password = 'Kjcs@12012024'

In [4]:
#encrypt the password using the key above
fernet = Fernet(key)
encrypted = fernet.encrypt(password.encode())
encrypted

b'gAAAAABpGVCu8873Nj0Cgz5-69heESv5c08kROgCrNEbZ1G858AbMuBPT3GIYHM_wK-Bct6IiaRhWVEx5HKIT5ZMugTZNQTogQ=='

In [5]:
#decrypt the password
decrypted = fernet.decrypt(encrypted.decode())
decrypted

b'Kjcs@12012024'

### ENCRYPTING THE FERNET KEY

In [38]:
master_key = key
master_key

b'NwMAISFRTA7chl4w7f8YCbZjL4bR9NuHBPkr_0UcX1s='

In [8]:
from Crypto.Cipher import AES
from Crypto.Protocol.KDF import PBKDF2
from Crypto.Random import get_random_bytes
import base64, json

In [ ]:
#Derive AES key from master password
#Generates 16 random bytes used as salt
salt = get_random_bytes(16)
#encrypts the password using PBKDF2 (Password-Based Key Derivation Function 2).
aes_key = PBKDF2(master_key, salt, dkLen=32, count=100_000)

In [25]:
salt

b'\xad\x83\xaf\xbcizrLtGr\xa0\xf2\xc9\xdbi'

In [ ]:
#Encrypt Fernet key with AES-EAX mode
cipher = AES.new(aes_key, AES.MODE_EAX)
#Encrypts the plaintext (fernet_key) using AES.
ciphertext, tag = cipher.encrypt_and_digest(master_key)

In [27]:
cipher.nonce

b'\xc00\xc6,r|5c\x1c\xe0Z5.\x0b(\x1f'

In [29]:
tag

b'\xe9\x8e\xf6Rz\x90\x98\xbf\xcf\xcc\xd7%"|\x00\xa9'

In [ ]:
#Package results for storage
#AES produces binary data, not human-readable text.
#So we base64-encode everything before saving to JSON.
encrypted_package = {
    "salt": base64.b64encode(salt).decode(),
    "nonce": base64.b64encode(cipher.nonce).decode(),
    "tag": base64.b64encode(tag).decode(),
    "ciphertext": base64.b64encode(ciphertext).decode()
}

In [22]:
encrypted_package

{'salt': 'rYOvvGl6ckx0R3Kg8snbaQ==',
 'nonce': 'wDDGLHJ8NWMc4Fo1LgsoHw==',
 'tag': '6Y72UnqQmL/PzNclInwAqQ==',
 'ciphertext': 'T6SEyMxD18cDJdnacfGtRBk0FnxDtpJ/0maNkTNCmd+UUZObM1x5qvXdF1c='}

### DECRYPTING THE FERNET KEY

In [23]:
#loading the encrypted credentials
salt = base64.b64decode(encrypted_package['salt'])
salt

b'\xad\x83\xaf\xbcizrLtGr\xa0\xf2\xc9\xdbi'

In [24]:
nonce = base64.b64decode(encrypted_package['nonce'])
nonce

b'\xc00\xc6,r|5c\x1c\xe0Z5.\x0b(\x1f'

In [28]:
tag = base64.b64decode(encrypted_package['tag'])
tag

b'\xe9\x8e\xf6Rz\x90\x98\xbf\xcf\xcc\xd7%"|\x00\xa9'

In [30]:
ciphertext = base64.b64decode(encrypted_package['ciphertext'])
ciphertext

b'O\xa4\x84\xc8\xccC\xd7\xc7\x03%\xd9\xdaq\xf1\xadD\x194\x16|C\xb6\x92\x7f\xd2f\x8d\x913B\x99\xdf\x94Q\x93\x9b3\\y\xaa\xf5\xdd\x17W'

In [31]:
aes_key = PBKDF2(master_key, salt, dkLen=32, count=100_000)

In [33]:
master_key

b'NwMAISFRTA7chl4w7f8YCbZjL4bR9NuHBPkr_0UcX1s='

In [35]:
cipher = AES.new(aes_key, AES.MODE_EAX, nonce)
fernet_key = cipher.decrypt_and_verify(ciphertext,tag)
fernet_key

b'NwMAISFRTA7chl4w7f8YCbZjL4bR9NuHBPkr_0UcX1s='

In [37]:
fernet_key1 = Fernet(fernet_key)
decrypted_key = fernet_key1.decrypt(encrypted.decode())
decrypted_key

b'Kjcs@12012024'

In [ ]:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
import base64

# -------------------------
# 1. Create key (32 bytes for AES-256)
# -------------------------
key = get_random_bytes(32)   # This is your AES key
print("AES Key:", base64.b64encode(key).decode())

# -------------------------
# 2. Encrypt message
# -------------------------
plaintext = "Hello World!"
plaintext_bytes = plaintext.encode()

cipher = AES.new(key, AES.MODE_EAX)
ciphertext, tag = cipher.encrypt_and_digest(plaintext_bytes)

print("Nonce:", base64.b64encode(cipher.nonce).decode())
print("Ciphertext:", base64.b64encode(ciphertext).decode())
print("Tag:", base64.b64encode(tag).decode())

# -------------------------
# 3. Decrypt message
# -------------------------
# (Pretend these values came from storage)
nonce = cipher.nonce
ciphertext = ciphertext
tag = tag

cipher_dec = AES.new(key, AES.MODE_EAX, nonce)
decrypted_bytes = cipher_dec.decrypt_and_verify(ciphertext, tag)

decrypted_text = decrypted_bytes.decode()
print("Decrypted:", decrypted_text)


In [ ]:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
import base64

# -------------------------
# AES Encrypt Function
# -------------------------
def aes_encrypt(plaintext):
    key = get_random_bytes(32)  # AES-256 key
    plaintext_bytes = plaintext.encode()

    cipher = AES.new(key, AES.MODE_EAX)
    ciphertext, tag = cipher.encrypt_and_digest(plaintext_bytes)

    return {
        "key": base64.b64encode(key).decode(),
        "nonce": base64.b64encode(cipher.nonce).decode(),
        "ciphertext": base64.b64encode(ciphertext).decode(),
        "tag": base64.b64encode(tag).decode()
    }


# -------------------------
# AES Decrypt Function
# -------------------------
def aes_decrypt(enc_data):
    key = base64.b64decode(enc_data["key"])
    nonce = base64.b64decode(enc_data["nonce"])
    ciphertext = base64.b64decode(enc_data["ciphertext"])
    tag = base64.b64decode(enc_data["tag"])

    cipher = AES.new(key, AES.MODE_EAX, nonce)
    decrypted_bytes = cipher.decrypt_and_verify(ciphertext, tag)

    return decrypted_bytes.decode()


# -------------------------
# TEST
# -------------------------
encrypted = aes_encrypt("Hello World!")
print("Encrypted Package:", encrypted)

decrypted = aes_decrypt(encrypted)
print("Decrypted:", decrypted)
